# Logistic Regression – Titanic Survival Prediction

**Name:** Ahmad Ali  
**Batch:** Data Science Weekday – Hyderabad  

In this assignment, we use the Titanic dataset to predict passenger survival using Logistic Regression. The assignment covers EDA, preprocessing, model building, evaluation, coefficient interpretation, and deployment using Streamlit.

## Importing Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, roc_curve,
                             confusion_matrix, classification_report)
from sklearn.preprocessing import StandardScaler
import pickle
import warnings
warnings.filterwarnings('ignore')

## Loading the Dataset

In [ ]:
train = pd.read_csv("Titanic_train.csv")
test  = pd.read_csv("Titanic_test.csv")

print("Train shape:", train.shape)
print("Test shape :", test.shape)
train.head()

## Data Exploration (EDA)

In [ ]:
train.info()
print("\nMissing Values:")
print(train.isnull().sum())
train.describe()

In [ ]:
# Survival count
plt.figure(figsize=(6,4))
sns.countplot(x='Survived', data=train, palette='Set2')
plt.title('Survival Distribution (0=No, 1=Yes)')
plt.show()
print(train['Survived'].value_counts())

In [ ]:
# Survival by Gender
plt.figure(figsize=(6,4))
sns.countplot(x='Sex', hue='Survived', data=train, palette='Set2')
plt.title('Survival by Gender')
plt.show()

Females had a significantly higher survival rate than males, reflecting the 'women and children first' policy during the disaster.

In [ ]:
# Survival by Passenger Class
plt.figure(figsize=(6,4))
sns.countplot(x='Pclass', hue='Survived', data=train, palette='Set1')
plt.title('Survival by Passenger Class')
plt.show()

In [ ]:
# Age distribution by survival
plt.figure(figsize=(8,5))
train[train['Survived']==1]['Age'].dropna().hist(alpha=0.6, label='Survived', bins=30, color='green')
train[train['Survived']==0]['Age'].dropna().hist(alpha=0.6, label='Did not survive', bins=30, color='red')
plt.title('Age Distribution by Survival')
plt.xlabel('Age')
plt.legend()
plt.show()

In [ ]:
# Fare vs Survival boxplot
plt.figure(figsize=(6,4))
sns.boxplot(x='Survived', y='Fare', data=train, palette='Set2')
plt.title('Fare Distribution by Survival')
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(10,7))
sns.heatmap(train.corr(numeric_only=True), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Feature Correlation Heatmap')
plt.show()

## Data Preprocessing

In [ ]:
# Fill missing Age with median
train['Age'].fillna(train['Age'].median(), inplace=True)

# Fill missing Embarked with mode
train['Embarked'].fillna(train['Embarked'].mode()[0], inplace=True)

# Drop Cabin (too many missing), Name, Ticket, PassengerId
train.drop(columns=['Cabin','Name','Ticket','PassengerId'], inplace=True)

# Encode categorical variables
train['Sex']      = train['Sex'].map({'male': 0, 'female': 1})
train['Embarked'] = train['Embarked'].map({'S': 0, 'C': 1, 'Q': 2})

print("Missing values after preprocessing:")
print(train.isnull().sum())
train.head()

## Defining Features and Target

In [ ]:
X = train.drop('Survived', axis=1)
y = train['Survived']

print("Features:", X.columns.tolist())
print("Shape:", X.shape)

## Train-Test Split and Scaling

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

print("Train size:", X_train.shape)
print("Test size :", X_test.shape)

## Building the Logistic Regression Model

In [ ]:
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("Model trained successfully!")

## Model Evaluation

In [ ]:
print("=== Model Performance ===")
print(f"Accuracy  : {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision : {precision_score(y_test, y_pred):.4f}")
print(f"Recall    : {recall_score(y_test, y_pred):.4f}")
print(f"F1 Score  : {f1_score(y_test, y_pred):.4f}")
print(f"ROC AUC   : {roc_auc_score(y_test, model.predict_proba(X_test)[:,1]):.4f}")
print()
print(classification_report(y_test, y_pred))

In [ ]:
# Confusion Matrix
plt.figure(figsize=(6,4))
sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

In [ ]:
# ROC Curve
y_prob = model.predict_proba(X_test)[:,1]
fpr, tpr, _ = roc_curve(y_test, y_prob)
auc_score = roc_auc_score(y_test, y_prob)

plt.figure(figsize=(7,5))
plt.plot(fpr, tpr, color='steelblue', label=f'ROC Curve (AUC = {auc_score:.3f})')
plt.plot([0,1],[0,1], 'k--', label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve – Titanic Survival Prediction')
plt.legend()
plt.show()

In [ ]:
# Cross Validation
cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')
print("Cross-Validation Scores:", cv_scores)
print(f"Mean CV Accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})") 

## Interpretation of Coefficients

In [ ]:
feature_names = train.drop('Survived', axis=1).columns
coeff_df = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': model.coef_[0]
}).sort_values('Coefficient', ascending=False)

print(coeff_df.to_string(index=False))

plt.figure(figsize=(8,5))
sns.barplot(x='Coefficient', y='Feature', data=coeff_df, palette='coolwarm')
plt.axvline(0, color='black', linewidth=0.8)
plt.title('Feature Coefficients – Logistic Regression')
plt.tight_layout()
plt.show()

**Interpretation:**
- **Sex (Female=1)** has the strongest positive coefficient — being female greatly increased survival chances
- **Pclass** has a negative coefficient — higher class number (lower class) reduced survival probability  
- **Fare** is positively correlated — passengers who paid more were more likely to survive
- **Age** has a slight negative coefficient — older passengers had slightly lower survival rates

## Deployment – Saving the Model

We save the trained model and scaler as pickle files. These are loaded by the Streamlit app for real-time predictions.

**Files for deployment:**
- `titanic_model.pkl` — Trained Logistic Regression model
- `titanic_scaler.pkl` — Fitted StandardScaler
- `app.py` — Streamlit web application
- `requirements.txt` — Python dependencies

**To run locally:**
```bash
pip install streamlit scikit-learn pandas numpy
streamlit run app.py
```

**For cloud deployment:**
1. Upload all files to GitHub repo
2. Go to https://streamlit.io/cloud
3. Connect GitHub repo and deploy

In [ ]:
# Save model and scaler
pickle.dump(model,  open("titanic_model.pkl",  "wb"))
pickle.dump(scaler, open("titanic_scaler.pkl", "wb"))

print("titanic_model.pkl  saved!")
print("titanic_scaler.pkl saved!")
print("Run the app using: streamlit run app.py")

## Interview Questions

### Q1. What is the difference between Precision and Recall?

**Precision** = True Positives / (True Positives + False Positives)
- Measures how many predicted survivors actually survived
- Important when cost of false alarms is high

**Recall** = True Positives / (True Positives + False Negatives)  
- Measures how many actual survivors were correctly identified
- In survival prediction, Recall is more critical — missing a survivor is worse than a false alarm

**Key difference:** Precision cares about prediction quality; Recall cares about coverage of actual positives.

---

### Q2. What is Cross-Validation and why is it important in Binary Classification?

**Cross-Validation** splits data into K folds, trains on K-1 folds and tests on 1 fold, repeating K times and averaging results.

**Why it is important:**
- Gives a reliable estimate of model performance on unseen data
- Prevents overfitting — model is tested on multiple different subsets
- Especially useful for imbalanced binary datasets like Titanic survival
- Helps compare and select the best model or hyperparameters
- Reduces variance in performance metrics compared to a single train-test split

## Conclusion

In this assignment, we built a Logistic Regression model to predict Titanic passenger survival.

Key findings:
- **Gender** was the strongest predictor — females had much higher survival rates
- **Passenger class** and **fare** were also significant predictors
- The model achieved strong accuracy and ROC-AUC score
- Cross-validation confirmed consistent performance across different data splits
- The model is deployed via Streamlit for real-time predictions